# 16 · Approved comprehension and appropriate-reliance study

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Run only with independently collected participant outcomes and a documented user-study decision. This is not a cancer-incidence study.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Load approved real data

In [ ]:
from oncoplate.governance import require_gate
require_gate(cfg,'user_study')
user_root=Path(cfg['root'])/'private/user_study'
data=read_table(user_root/'outcomes.csv')
for c in ['comprehension_correct','appropriate_reliance','elapsed_seconds']:data[c]=data[c].astype(float)
print(data.groupby('arm').participant_id.nunique())

## 2. Analyse the registered parallel-arm comparison

In [ ]:
from oncoplate.studies import user_study_analysis
result=user_study_analysis(data,B=1000,seed=0)
write_json(p['reports']/'user_study_analysis.json',result);print(json.dumps(result,indent=2))

## 3. Report limits and protocol fidelity

In [ ]:
report={'measures':['comprehension','appropriate_reliance','elapsed_seconds'],'unit':'participant',
'cancer_incidence_measured':False,'clinical_patients_assumed':False,'formative_participants_included_in_confirmatory_analysis':False,
'protocol_review_reference_required':True}
write_json(p['reports']/'user_study_reporting_contract.json',report)
print('Keep actual participant identifiers and row-level data private; export reviewed aggregates only.')

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
